# NeXo v3.0 — RAT Underservice Detection (XGBoost)

## Methodology: Gradient Boosting for Binary Classification

This notebook implements **Phase 5-6** for the RAT underservice use case:
- Predict whether a subscriber is **underserved** by their network RAT
- Target: `rat_gap_score > 0.5` (device capability >> actual RAT)

### Why XGBoost?
| Criterion | LightGBM (CEM) | XGBoost (RAT) |
|-----------|----------------|---------------|
| Regularization | L1/L2 built-in | **Stronger L1/L2 + tree pruning** |
| Handling imbalance | Good | **Better with scale_pos_weight** |
| Monotonic constraints | Limited | **Native support** |
| Academic citations | High | **Higher (most cited GB lib)** |

For a binary classification defense problem, XGBoost's regularization and  
`scale_pos_weight` parameter make it robust on imbalanced telecom data.

### Data
- 500K real BSS subscribers (March 2026)
- 9 features: usage, attach SRs, NE index, area QoS
- Target: 9.18% underserved rate

In [ ]:
# --- Phase 0: Imports ---
import os
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

SEED = 42
np.random.seed(SEED)
ARTIFACT_DIR = "data"
MODEL_DIR = "models"

print(f"[{datetime.now():%H:%M:%S}] RAT Underservice Training Started")
print(f"XGBoost: {xgb.__version__}")

## Phase 4: Load Dataset

In [ ]:
rat_data = np.load(f"{ARTIFACT_DIR}/rat_underservice_mar2026.npz", allow_pickle=True)
X = rat_data["X"].astype(np.float32)
y = rat_data["y"].astype(np.int64)
feature_names = list(rat_data["feature_names"])

print(f"Dataset loaded: X={X.shape}, y={y.shape}")
print(f"Features: {feature_names}")
print(f"Underservice rate: {y.mean()*100:.2f}%")

# Train/val/test split (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"\nSplit sizes:")
print(f"  Train: {len(y_train):,} (underserved: {y_train.sum():,})")
print(f"  Val:   {len(y_val):,} (underserved: {y_val.sum():,})")
print(f"  Test:  {len(y_test):,} (underserved: {y_test.sum():,})")

## Phase 5A: Baseline XGBoost
Handle class imbalance with `scale_pos_weight = neg / pos`.

In [ ]:
scale_pos_weight = float((y_train == 0).sum()) / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

print("\n" + "=" * 60)
print("Training Baseline: XGBoost Classifier")
print("=" * 60)

xgb_base = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="logloss",
)

xgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

# Evaluate
def report(name, y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba) if y_proba is not None else None
    print(f"\n{name}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    if auc:
        print(f"  ROC-AUC:   {auc:.4f}")
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": auc}

y_pred_base_val = xgb_base.predict(X_val)
y_proba_base_val = xgb_base.predict_proba(X_val)[:, 1]
y_pred_base_test = xgb_base.predict(X_test)
y_proba_base_test = xgb_base.predict_proba(X_test)[:, 1]

metrics_base_val = report("XGB Baseline Validation", y_val, y_pred_base_val, y_proba_base_val)
metrics_base_test = report("XGB Baseline Test", y_test, y_pred_base_test, y_proba_base_test)

### Baseline Feature Importances

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
importance = pd.Series(xgb_base.feature_importances_, index=feature_names).sort_values()
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("XGBoost Feature Importances (Baseline)")
ax.set_xlabel("Gain")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/rat_xgb_importance_baseline.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/rat_xgb_importance_baseline.png")

## Phase 5B: Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
print("\n" + "=" * 60)
print("Hyperparameter Tuning: RandomizedSearchCV")
print("=" * 60)

param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2, 0.5],
}

xgb_search = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="logloss",
)

random_search = RandomizedSearchCV(
    xgb_search,
    param_distributions,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)

print(f"\nBest params: {random_search.best_params_}")
print(f"Best CV F1: {random_search.best_score_:.4f}")

### Evaluate Tuned Model

In [ ]:
xgb_tuned = random_search.best_estimator_

y_pred_tuned_val = xgb_tuned.predict(X_val)
y_proba_tuned_val = xgb_tuned.predict_proba(X_val)[:, 1]
y_pred_tuned_test = xgb_tuned.predict(X_test)
y_proba_tuned_test = xgb_tuned.predict_proba(X_test)[:, 1]

metrics_tuned_val = report("XGB Tuned Validation", y_val, y_pred_tuned_val, y_proba_tuned_val)
metrics_tuned_test = report("XGB Tuned Test", y_test, y_pred_tuned_test, y_proba_tuned_test)

## Phase 6A: Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Baseline": [metrics_base_test["acc"], metrics_base_test["prec"], 
                 metrics_base_test["rec"], metrics_base_test["f1"], metrics_base_test["auc"]],
    "Tuned": [metrics_tuned_test["acc"], metrics_tuned_test["prec"], 
              metrics_tuned_test["rec"], metrics_tuned_test["f1"], metrics_tuned_test["auc"]],
}, index=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])

print("\n" + "=" * 60)
print("MODEL COMPARISON (Test Set)")
print("=" * 60)
print(comparison.round(4))

comparison.to_csv(f"{ARTIFACT_DIR}/rat_model_comparison.csv")
print(f"Saved: {ARTIFACT_DIR}/rat_model_comparison.csv")

## Phase 6B: Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Served", "Underserved"], yticklabels=["Served", "Underserved"])
axes[0].set_title("Confusion Matrix (Tuned)")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# ROC Curve
fpr_base, tpr_base, _ = roc_curve(y_test, y_proba_base_test)
fpr_tuned, tpr_tuned, _ = roc_curve(y_test, y_proba_tuned_test)
axes[1].plot(fpr_base, tpr_base, label=f"Baseline (AUC={metrics_base_test['auc']:.3f})", lw=2)
axes[1].plot(fpr_tuned, tpr_tuned, label=f"Tuned (AUC={metrics_tuned_test['auc']:.3f})", lw=2)
axes[1].plot([0, 1], [0, 1], "r--", lw=1, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Feature importance (tuned)
imp_tuned = pd.Series(xgb_tuned.feature_importances_, index=feature_names).sort_values()
imp_tuned.plot(kind="barh", ax=axes[2], color="coral")
axes[2].set_title("Feature Importance (Tuned)")
axes[2].set_xlabel("Gain")

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/rat_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/rat_evaluation.png")

## Phase 6C: Classification Report

In [ ]:
print("\nDetailed Classification Report (Tuned Model, Test Set):")
print(classification_report(y_test, y_pred_tuned_test, target_names=["Served", "Underserved"]))

## Phase 7: Save Model & Register

In [ ]:
# Save tuned model
model_path = f"{MODEL_DIR}/rat_underservice_v3_xgb.joblib"
joblib.dump(xgb_tuned, model_path)
print(f"Saved model: {model_path} ({os.path.getsize(model_path)/1e6:.1f} MB)")

# Save baseline too
joblib.dump(xgb_base, f"{MODEL_DIR}/rat_underservice_v3_xgb_baseline.joblib")
print(f"Saved baseline: {MODEL_DIR}/rat_underservice_v3_xgb_baseline.joblib")

# Save feature names
joblib.dump(feature_names, f"{MODEL_DIR}/rat_v3_feature_names.joblib")
print(f"Saved feature names: {MODEL_DIR}/rat_v3_feature_names.joblib")

# Model card
model_card = f"""
# RAT Underservice Detection Model Card (v3.0)

## Model Details
- **Algorithm**: XGBoost Classifier
- **Version**: v3.0
- **Training Date**: {datetime.now().isoformat()}
- **Dataset**: Real BSS subscribers, March 2026
- **Samples**: {len(y):,} subscribers
- **Features**: {len(feature_names)} ({', '.join(feature_names)})
- **Class imbalance**: scale_pos_weight = {scale_pos_weight:.2f}

## Performance (Test Set)
- **Accuracy**:  {metrics_tuned_test['acc']:.4f}
- **Precision**: {metrics_tuned_test['prec']:.4f}
- **Recall**:    {metrics_tuned_test['rec']:.4f}
- **F1**:        {metrics_tuned_test['f1']:.4f}
- **ROC-AUC**:   {metrics_tuned_test['auc']:.4f}

## Best Hyperparameters
{random_search.best_params_}

## Feature Importance (Top 5)
"""

for i, (feat, imp) in enumerate(imp_tuned.tail(5).items(), 1):
    model_card += f"{i}. {feat} (gain = {imp:.4f})\n"

with open(f"{MODEL_DIR}/rat_underservice_v3_model_card.md", "w") as f:
    f.write(model_card)
print(f"Saved model card: {MODEL_DIR}/rat_underservice_v3_model_card.md")

## Summary

| Metric | Baseline | Tuned | Δ |
|--------|----------|-------|---|
| Accuracy | {metrics_base_test['acc']:.4f} | {metrics_tuned_test['acc']:.4f} | +{metrics_tuned_test['acc']-metrics_base_test['acc']:.4f} |
| F1 | {metrics_base_test['f1']:.4f} | {metrics_tuned_test['f1']:.4f} | +{metrics_tuned_test['f1']-metrics_base_test['f1']:.4f} |
| ROC-AUC | {metrics_base_test['auc']:.4f} | {metrics_tuned_test['auc']:.4f} | +{metrics_tuned_test['auc']-metrics_base_test['auc']:.4f} |

**Artifacts:**
- `models/rat_underservice_v3_xgb.joblib` — winning model
- `models/rat_underservice_v3_xgb_baseline.joblib` — baseline
- `models/rat_v3_feature_names.joblib` — feature list
- `models/rat_underservice_v3_model_card.md` — documentation

In [ ]:
print(f"[{datetime.now():%H:%M:%S}] RAT Underservice Training Complete")